# Simulations (C++/Python switchable) — VCM 3.4

Variant of Simulations.ipynb with a Python/C++ backend selector (see section 0). Production simulation runs: Brazilian-disk + crack-network coupling, every `disk_*` case from this directory.

Set `CASE` in the next cell and run the notebook top to bottom; the
dispatcher at the bottom calls the matching `run_<case>()` function.

Available cases (matching the per-case notebooks in this directory):
- `disk_compression_2`
- `disk_compression_inclined`
- `disk_energetics`
- `disk_experiments`

Each `run_<case>()` is a faithful inlining of the corresponding notebook's
code cells. Markdown cells from the originals become the function docstring.
If a case notebook changes, regenerate this file with `_build_runners.py`.


## 0. Backend selector (Python vs C++)

Pick the computation backend used for the crack-side KKT solve and the
topology intersection detector. The BEM-side (`ensure_bem_field`)
currently runs Python in both configurations; only the parts that have
been ported to C++/OpenMP via the `bem_cpp` extension are affected.

| ENGINE  | What runs in C++                                                   |
|---------|--------------------------------------------------------------------|
| 'python'| Nothing -- pure-Python reference (same as Simulations.ipynb)       |
| 'cpp'   | KKT crack-solve assemblers + KKT solve + segment-intersection detect |

The two engines produce results that agree to machine precision
(~1e-12 relative) on every case verified so far. The C++ engine
gives 16-27x full-solve speedup on networks with 100-1000 panels
(see VCEM_4_0/cpp/python/bench_kkt_scaling.py).

In [ ]:
# Backend for the crack-side KKT solve and intersection detection.
ENGINE = 'cpp'    # options: 'python' | 'cpp'


## 1. Case selector


In [ ]:
# Pick which workflow to run. Must be one of the keys in CASES at the bottom.
CASE = "disk_compression_2"  # options: disk_compression_2 | disk_compression_inclined | disk_energetics | disk_experiments

# Safety guard: keep DRY_RUN = True the first time you open this notebook so
# Run-All only defines the case functions without executing them. Set to
# False to actually invoke run_<CASE>() at the bottom.
DRY_RUN = False


## 2. Common preamble (imports + repo path)


In [ ]:
from __future__ import annotations  # hoisted from per-case notebooks

from pathlib import Path
import os, sys
import numpy as np  # noqa: F401  (every case imports it)
import matplotlib.pyplot as plt  # noqa: F401  (most cases need it)

nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Several legacy notebooks did `from preamble import *` at the top of every
# cell, which is illegal inside a function body. Hoist that wildcard import
# here at module scope so every case function inherits the same namespace,
# and the per-case bodies (stripped of their own `from preamble import *`
# lines by _build_runners.py) continue to work unchanged.
from preamble import *  # noqa: F401,F403

print(f'Running case: {CASE!r}')
print(f'Repo root:    {repo_root}')


# ── Backend dispatch (re-runnable: no Kernel-Restart needed after edits) ──
# This block force-reloads every module we monkey-patch and re-installs the
# patches every time the cell runs. That way, after any code change to the
# patched modules you can just re-run the preamble cell instead of
# Restart-Kernel-and-Run-All.
import sys as _sys, importlib as _il

# Add VCEM_4_0/cpp/python to sys.path so `import bem_cpp` resolves. The
# original preamble only adds repo_root (= VCEM_4_0).
_cpp_pkg_dir = str(repo_root / 'cpp' / 'python')
if _cpp_pkg_dir not in _sys.path:
    _sys.path.insert(0, _cpp_pkg_dir)

# Try to import bem_cpp, fall back to engine='python' if missing.
# Drop any half-baked import from a previous run so reload() can't fail
# on a missing-submodule cache entry.
for _stale in [m for m in list(_sys.modules) if m == 'bem_cpp' or m.startswith('bem_cpp.')]:
    del _sys.modules[_stale]
_CPP_AVAILABLE = False
_bem_cpp_err = None
try:
    import bem_cpp  # noqa: F401
    _CPP_AVAILABLE = True
except Exception as _e:
    _bem_cpp_err = _e
    if ENGINE == 'cpp':
        ENGINE = 'python'

# Force-reload every patched module so a previous run's stale `_orig_*`
# closure can't leak across edits. We capture _orig_* AFTER the reload.
import fracture_utils.Usolver.parametrization as _para_mod
import fracture_utils.Ugenerator.generator as _gen_mod
import fracture_utils.Ubem.brazilian_disk_bem as _bdb_mod
import fracture_utils.Ubem.disk_iterative_coupling as _dic_mod
for _mod in (_para_mod, _gen_mod, _bdb_mod, _dic_mod):
    _il.reload(_mod)
# Re-bind after reload (the names above now point at the old module
# objects; the reloaded ones are returned by _il.reload and stored back
# in sys.modules, but we want the fresh attributes here).
import fracture_utils.Usolver.parametrization as _para_mod
import fracture_utils.Ugenerator.generator as _gen_mod
import fracture_utils.Ubem.brazilian_disk_bem as _bdb_mod
import fracture_utils.Ubem.disk_iterative_coupling as _dic_mod

_orig_solve = _para_mod.DCENetworkStaticV4.solve
def _solve_with_engine(self, *args, **kwargs):
    kwargs.setdefault('engine', ENGINE)
    return _orig_solve(self, *args, **kwargs)
_para_mod.DCENetworkStaticV4.solve = _solve_with_engine

_orig_det = _gen_mod.CrackNetworkGenerator.detect_and_split_intersections
def _det_with_engine(self, *args, **kwargs):
    kwargs.setdefault('engine', ENGINE)
    return _orig_det(self, *args, **kwargs)
_gen_mod.CrackNetworkGenerator.detect_and_split_intersections = _det_with_engine

_orig_ensure  = _bdb_mod.ensure_bem_field
_orig_compute = _bdb_mod.compute_bem_brazilian_disk_field
def _ensure_with_engine(*args, **kwargs):
    kwargs.setdefault('engine', ENGINE)
    return _orig_ensure(*args, **kwargs)
def _compute_with_engine(*args, **kwargs):
    kwargs.setdefault('engine', ENGINE)
    return _orig_compute(*args, **kwargs)
_bdb_mod.ensure_bem_field = _ensure_with_engine
_bdb_mod.compute_bem_brazilian_disk_field = _compute_with_engine

_orig_sbwet = _dic_mod.solve_bem_with_extra_boundary_tractions
def _sbwet_with_engine(*args, **kwargs):
    kwargs.setdefault('engine', ENGINE)
    return _orig_sbwet(*args, **kwargs)
_dic_mod.solve_bem_with_extra_boundary_tractions = _sbwet_with_engine

# Loud banner so the user can't miss which backend is actually live.
print()
print('=' * 62)
print(f' ACTIVE ENGINE = {ENGINE.upper():<6s} '
      + ('(C++/OpenMP, %d threads)' % bem_cpp.openmp_max_threads() if _CPP_AVAILABLE and ENGINE == 'cpp'
         else '(pure Python)'))
if ENGINE == 'cpp':
    print(' Patched: DCENetworkStaticV4.solve, detect_and_split_intersections,')
    print('          ensure_bem_field, compute_bem_brazilian_disk_field,')
    print('          solve_bem_with_extra_boundary_tractions')
    print(f' bem_cpp v{bem_cpp.__version__} from {bem_cpp.__file__}')
else:
    if _bem_cpp_err is not None:
        print(f' Note: bem_cpp import failed: {_bem_cpp_err!r}')
        print(f'       (looked in {_cpp_pkg_dir})')
        # Detailed diagnostic so the root cause is obvious.
        print()
        print(' --- diagnostic ---')
        print(f'   sys.executable = {_sys.executable}')
        print(f'   sys.version    = {_sys.version.split(chr(10))[0]}')
        _pkg = _cpp_pkg_dir + '/bem_cpp'
        import os as _os
        if _os.path.isdir(_pkg):
            _files = sorted(_os.listdir(_pkg))
            print(f'   {_pkg} contents:')
            for _f in _files:
                print(f'     {_f}')
            _pyd = [f for f in _files if f.endswith('.pyd') or f.endswith('.so')]
            if _pyd:
                # cp310-win_amd64 -> Python 3.10. Check vs running interp.
                _v = _sys.version_info
                _tag = f'cp{_v.major}{_v.minor}'
                _matching = [f for f in _pyd if _tag in f]
                if not _matching:
                    print(f'   *** ABI MISMATCH: extension is {_pyd[0]}, '
                          f'but running interpreter is {_tag} (Python '
                          f'{_v.major}.{_v.minor}). ***')
                    print('   *** This usually means Jupyter is using a '
                          'different Python than the conda vcem_4_0 env. ***')
                    print('   *** Pick the vcem_4_0 kernel in Jupyter '
                          '(Kernel -> Change Kernel) or rebuild the '
                          'extension for your running Python ***')
                    print('   *** (cd VCEM_4_0/cpp/build && cmake --build . '
                          '--config Release --target bem_cpp && cmake '
                          '--install . --config Release). ***')
        else:
            print(f'   *** directory missing: {_pkg}')
            print('   *** rebuild via VCEM_4_0/cpp/README.md ***')
print('=' * 62)
import sys as _sys2; _sys2.stdout.flush()


## 3. Per-case workflows

Each cell below defines `run_<case>()`. Only the function matching `CASE`
is invoked by the dispatcher; the others are defined but not executed.


### Case `disk_compression_2` (from `disk_compression_2.ipynb`)


In [ ]:
def run_disk_compression_2():
    """Case extracted from disk_compression_2.ipynb."""
    global BEMBlockV2, BrazilianDiskParams, CrackGrowthParams, Path, PlotParams, build_boundary, compose_solver_kwargs_v2, compute_crack_boundary_tractions_direct, ensure_bem_field, make_standard_plot_hook, np, os, plot_total_field, run_network_growth_uncoupled, solve_bem_with_extra_boundary_tractions
    from pathlib import Path
    import os, sys
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
    from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
    from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.boundary_conditions import build_boundary

    # -----------------------
    # Output + mode selection
    # -----------------------
    bem_dir = repo_root / "output" / "BEM_Brazilian_disk_iterative"
    bem_dir.mkdir(parents=True, exist_ok=True)

    # Choose one:
    #  - "no_coupling"
    #  - "iterative"  (outer-cycle updates only)
    #  - "direct"     (outer-cycle updates only)
    COUPLING_METHOD = "iterative"
    OUTER_CYCLES = 2

    # -----------------------
    # Network (YOU control)
    # -----------------------
    mm = 1e-3
    vertices = np.array([
        [0, -2.0*mm, -3.0*mm],
        [1,  2.0*mm,  3.0*mm],
    ], float)
    connectivity = np.array([[0, 1]], int)

    # -----------------------
    # (1) BEM baseline
    # -----------------------
    SKIP_BEM_SOLVE = False  # set True to skip BEM solve and load from disk (if available)
    disk = BrazilianDiskParams(R=25.4e-3/2, P_total=3.8e3, E=231.52e9, nu=0.3, n_boundary_elements=120, n_grid=120)

    bem_block = BEMBlockV2(disk=disk, out_dir=bem_dir)
    ensure_bem_field(
        disk,
        
        bem_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=True,
        save_contours=True,
    )

    boundary_mesh = build_boundary({"type": "circle", "R": disk.R, "n_boundary": disk.n_boundary_elements, "center": (0.0, 0.0)})

    # -----------------------
    # Plot options
    # -----------------------
    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=bem_dir,
        combined_out_dir=bem_dir,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0,2), int)
        return V, C


    def _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot):
        xs = np.load(bem_dir / "xs.npy")
        ys = np.load(bem_dir / "ys.npy")
        np.save(bem_dir / "xs.npy", xs)
        np.save(bem_dir / "ys.npy", ys)
        np.save(bem_dir / "Sxx.npy", np.asarray(Sxx_tot, float))
        np.save(bem_dir / "Syy.npy", np.asarray(Syy_tot, float))
        np.save(bem_dir / "Sxy.npy", np.asarray(Sxy_tot, float))


    # One-way growth kwargs (used for propagation increments)
    one_way_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs={"n_crack_elements": 40})

    # Direct kwargs (used only at outer-cycle coupling checkpoints)
    aug_payload = bem_block.build_augmented_payload(d_mode="correction_zero", gauss_n=4)
    direct_kwargs = compose_solver_kwargs_v2(
        "full_kkt",
        base_solver_kwargs={
            "augmented_enforce_crack_equilibrium": True,
            "augmented_equilibrate": True,
            "augmented_equilibrate_cols": False,
            "augmented_physical_scaling": True,
            "augmented_scale_traction_rows": True,
        },
        augmented_payload=aug_payload,
        augmented_ridge_q=1e-4,
        augmented_ridge_y=1e-8,
        augmented_keep_bem_applied=True,
    )

    V_curr = vertices.copy()
    C_curr = connectivity.copy()
    res_last = None

    for cyc in range(1, int(OUTER_CYCLES) + 1):
        print(f"\n[outer-cycle] {cyc}/{OUTER_CYCLES} method={COUPLING_METHOD}")

        # Growth for ONE outer cycle using one-way applied field from current bem_dir arrays
        grow = CrackGrowthParams(max_cycles=5, L_limit_mm=20.0, vertex_high=18, solver_kwargs=one_way_kwargs)
        out_growth = bem_dir / f"outer_{cyc:02d}_growth"
        out_growth.mkdir(parents=True, exist_ok=True)

        res_growth = run_network_growth_uncoupled(
            bem_dir=bem_dir,
            out_dir=out_growth,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        # Outer-cycle coupling update (only here)
        if COUPLING_METHOD == "iterative":
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_growth, boundary_mesh=boundary_mesh)
            solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=bem_dir,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                show=False,
            )
            res_last = res_growth

        elif COUPLING_METHOD == "direct":
            # Solve direct KKT on current network (no additional growth), then update bem_dir arrays
            grow_direct = CrackGrowthParams(max_cycles=0, L_limit_mm=20.0, vertex_high=18, solver_kwargs=direct_kwargs)
            out_direct = bem_dir / f"outer_{cyc:02d}_direct"
            out_direct.mkdir(parents=True, exist_ok=True)

            Vg, Cg = _network_to_arrays(res_growth.calc.network)
            res_direct = run_network_growth_uncoupled(
                bem_dir=bem_dir,
                out_dir=out_direct,
                vertices=Vg,
                connectivity=Cg,
                params=grow_direct,
                plot_hook=None,
            )

            Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
                bem_dir=bem_dir,
                res=res_direct,
                out_dir=out_direct,
                tag=f"outer_{cyc:02d}_direct",
                params=plot_params,
                show=True,
            )
            _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot)
            res_last = res_direct

        else:
            # no_coupling
            res_last = res_growth

        # advance network for next outer cycle from growth result
        V_curr, C_curr = _network_to_arrays(res_growth.calc.network)

    print("\n[done] outer-cycle run completed")

    return locals()


### Case `disk_compression_inclined` (from `disk_compression_inclined.ipynb`)


In [ ]:
def run_disk_compression_inclined():
    """Case extracted from disk_compression_inclined.ipynb."""
    global BrazilianDiskParams, CrackGrowthParams, Path, PlotParams, ensure_bem_field, make_standard_plot_hook, np, os, run_network_growth_uncoupled
    from pathlib import Path
    import os, sys
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import (BrazilianDiskParams, ensure_bem_field)
    from fracture_utils.Ubem.disk_network_propagation import (CrackGrowthParams, run_network_growth_uncoupled)
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook

    bem_dir   = repo_root / "output" / "BEM_Brazilian_disk"
    # crack_dir = repo_root / "output" / "KinkedCrackPropagation"
    # combined_dir = bem_dir / "combined_uncoupled"

    # -----------------------
    # Network (YOU control)
    # -----------------------
    mm = 1e-3
    vertices = np.array([
        [0, -2.0*mm, -3.0*mm],
        [1,  2.0*mm,  3.0*mm],
    ], float)

    connectivity = np.array([[0, 1]], int)

    # -----------------------
    # (1) BEM (fast reuse option)
    # -----------------------
    SKIP_BEM_SOLVE = True   # <-- set False to recompute
    disk = BrazilianDiskParams(R=25.4e-3/2, P_total=3.8e3, E=231.52e9, nu=0.3, n_boundary_elements=60, n_grid=120)

    ensure_bem_field(
        disk,
        bem_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=True,              # only used if recompute=True
        save_contours=True,     # only used if recompute=True
    )

    # -----------------------
    # (2) Propagation + TOTAL-only plotting
    # -----------------------

    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,   
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=bem_dir,
        combined_out_dir=bem_dir,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )


    grow = CrackGrowthParams(max_cycles=5, L_limit_mm=20.0, vertex_high=18)

    res_final = run_network_growth_uncoupled(
        bem_dir=bem_dir,
        out_dir=bem_dir,
        vertices=vertices,
        connectivity=connectivity,
        params=grow,
        plot_hook=hook,
    )

    return locals()


### Case `disk_energetics` (from `disk_energetics.ipynb`)


In [ ]:
def run_disk_energetics():
    """Case extracted from disk_energetics.ipynb."""
    global BEMSolver2D, BrazilianDiskParams, ContourOpts, MISSING, Path, PlotParams, build_bem_dense_operators, build_boundary, circle_inside, compute_crack_boundary_tractions_direct, compute_disk_boundary_compliance_from_solver, datetime, dnp, eval_and_plot_stress_components_contours_separate, fields, importlib, json, make_standard_plot_hook, np, os, plt, re, shutil, solve_bem_with_extra_boundary_tractions, subprocess
    from pathlib import Path
    import os, sys
    import json
    import shutil
    import subprocess
    from datetime import datetime
    from dataclasses import fields, MISSING
    import numpy as np
    import matplotlib.pyplot as plt

    # -----------------------
    # Repo import path
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams
    from fracture_utils.Ubem.boundary_conditions import build_boundary
    from fracture_utils.Ubem.bem_solver import BEMSolver2D
    from fracture_utils.Ubem.bem_stress_field import circle_inside
    from fracture_utils.Ubem.bem_stress_plotter import (
        ContourOpts,
        eval_and_plot_stress_components_contours_separate,
    )
    from fracture_utils.Ubem.direct_kkt_coupling import build_bem_dense_operators
    from fracture_utils.Ubem.crack_energetics import compute_disk_boundary_compliance_from_solver
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook

    import importlib
    import fracture_utils.Ubem.disk_network_propagation as dnp
    importlib.reload(dnp)
    CrackGrowthParams = dnp.CrackGrowthParams
    run_network_growth_uncoupled = dnp.run_network_growth_uncoupled


    # -----------------------
    # User config
    # -----------------------
    mm = 1e-3

    # Displacement-controlled loading (borrowed from BEM_displacement.ipynb)
    U_Y_TOP_IMPOSED = -1e-5
    U_Y_BOTTOM_IMPOSED = 0.0

    # Disk + material params (same family as previous notebooks)
    disk = BrazilianDiskParams(
        R=25.4e-3 / 2,
        P_total=3.8e3,     # kept for compatibility (not used in imposed_uy solve)
        E=231.52e9,
        nu=0.3,
        n_boundary_elements=90,
        
        gauss_n=6,
        plane_strain=True,
        arc_half_angle_deg=15.0,
        h=6.35e-3,
        n_grid=120,
        pad_frac=0.03,
    )

    # Crack network + growth setup (borrowed from disk_compression_inclined.ipynb)
    Kc_demo = 0.0e6
    vertices = np.array([
        [0, -1.0*mm, -6.0*mm],
        [1, -0.5*mm,  5.0*mm],
        [2,  1.0*mm, -3.0*mm],
        [3,  1.5*mm,  2.5*mm],
        [4, -2.0*mm, -2.5*mm],
        [5,  4.0*mm, -1.0*mm],
        [6, -6.0*mm,  3.0*mm],
        [7,  2.0*mm,  1.5*mm],
    ], float)
    connectivity = np.array([
        [0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
    ], int)

    def _git_short_hash(default: str = "nogit") -> str:
        try:
            out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root)
            return out.decode("utf-8", errors="ignore").strip() or default
        except Exception:
            return default


    def _run_id() -> str:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{ts}_{_git_short_hash()}"


    def _dataclass_defaults(dc_type):
        d = {}
        for f in fields(dc_type):
            if f.default is not MISSING:
                d[f.name] = f.default
            elif f.default_factory is not MISSING:  # type: ignore[attr-defined]
                d[f.name] = f.default_factory()      # type: ignore[misc]
            else:
                d[f.name] = None
        return d


    def _fmt(v):
        if isinstance(v, np.ndarray):
            return np.array2string(v, separator=", ")
        if isinstance(v, (list, tuple, dict)):
            try:
                return json.dumps(v, ensure_ascii=False)
            except Exception:
                return str(v)
        return str(v)


    def _write_provenance(run_dir: Path, run_id: str, table_rows):
        prov = run_dir / f"provenance_{run_id}.md"
        lines = []
        lines.append(f"# Provenance - {run_id}\n")
        lines.append("\n")
        lines.append(f"- Generated: {datetime.now().isoformat()}\n")
        lines.append(f"- Repo root: `{repo_root}`\n")
        lines.append(f"- Notebook: `VCM_3_4/notebooks/disk_energetics.ipynb`\n")
        lines.append("\n")
        lines.append("## Run Configuration\n")
        lines.append("\n")
        lines.append("| Group | Parameter | Default | Effective | Source |\n")
        lines.append("|---|---|---|---|---|\n")
        for g, p, d, e, s in table_rows:
            lines.append(f"| {g} | `{p}` | `{_fmt(d)}` | `{_fmt(e)}` | {s} |\n")
        prov.write_text("".join(lines), encoding="utf-8")


    def _write_provenance_json(run_dir: Path, run_id: str, table_rows):
        prov_json = run_dir / f"provenance_{run_id}.json"
        rows = []
        for g, p, d, e, s in table_rows:
            rows.append({
                "group": g,
                "parameter": p,
                "default": d,
                "effective": e,
                "source": s,
            })
        payload = {
            "run_id": run_id,
            "generated_at": datetime.now().isoformat(),
            "repo_root": str(repo_root),
            "notebook": "VCM_3_4/notebooks/disk_energetics.ipynb",
            "rows": rows,
        }

        def _json_default(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, (np.floating, np.integer)):
                return obj.item()
            return str(obj)

        prov_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")


    run_id = _run_id()
    run_dir = repo_root / "output" / "NetworkSimulations" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"[run_id] {run_id}")
    print(f"[run_dir] {run_dir}")

    OUT_DIR = run_dir
    BEM_DIR = run_dir
    RUN_DIR = run_dir
    BEM_DIR.mkdir(parents=True, exist_ok=True)
    RUN_DIR.mkdir(parents=True, exist_ok=True)


    # -----------------------
    # Displacement-controlled BEM solve
    # -----------------------
    def _build_disk_mesh(params: BrazilianDiskParams):
        return build_boundary({"type": "circle", "R": float(params.R), "n_boundary": int(params.n_boundary_elements), "center": (0.0, 0.0)})


    def _arc_masks(mesh, params: BrazilianDiskParams):
        theta_deg = np.asarray(mesh.theta_deg, dtype=float)
        arc_half = float(params.arc_half_angle_deg)
        top = (theta_deg >= 90.0 - arc_half) & (theta_deg <= 90.0 + arc_half)
        bot = (theta_deg >= -90.0 - arc_half) & (theta_deg <= -90.0 + arc_half)
        return top, bot


    def solve_disk_imposed_uy(params: BrazilianDiskParams, uy_top: float, uy_bottom: float = 0.0):
        mesh = _build_disk_mesh(params)
        top, bot = _arc_masks(mesh, params)
        n = int(mesh.n_seg)

        solver = BEMSolver2D(E=float(params.E), nu=float(params.nu), h=float(params.h), plane_strain=bool(params.plane_strain))
        for i in range(n):
            solver.add_element(
                float(mesh.x1[i]), float(mesh.y1[i]),
                float(mesh.x2[i]), float(mesh.y2[i]),
                True, 0.0, 0.0,
            )

        CplusH, G = build_bem_dense_operators(
            segments=solver.segs,
            E=float(params.E),
            nu=float(params.nu),
            plane_strain=bool(params.plane_strain),
            gauss_n=int(params.gauss_n),
        )

        C_bem = np.hstack([CplusH, -G])
        C_bc = np.zeros((2 * n, 4 * n), float)
        d = np.zeros((2 * n,), float)

        for i in range(n):
            r0 = 2 * i
            r1 = 2 * i + 1

            # keep u_x free by enforcing t_x = 0
            C_bc[r0, 2 * n + 2 * i + 0] = 1.0

            if top[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_top)
            elif bot[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_bottom)
            else:
                C_bc[r1, 2 * n + 2 * i + 1] = 1.0

        A = np.vstack([C_bem, C_bc])
        b = np.concatenate([np.zeros((2 * n,), float), d])
        y, *_ = np.linalg.lstsq(A, b, rcond=None)

        u = y[:2 * n]
        t = y[2 * n:]
        solver.u_x = np.asarray(u[0::2], float).copy()
        solver.u_y = np.asarray(u[1::2], float).copy()
        solver.t_x = np.asarray(t[0::2], float).copy()
        solver.t_y = np.asarray(t[1::2], float).copy()
        solver._colloc_xy = np.column_stack([np.asarray(mesh.xm, float), np.asarray(mesh.ym, float)])

        return solver, mesh


    def save_bem_field_arrays_and_plots(solver, params: BrazilianDiskParams, out_dir: Path, basename: str = "brazilian_disk"):
        bbox = (-params.R, params.R, -params.R, params.R)
        inside = circle_inside(float(params.R), center=(0.0, 0.0), pad=float(params.pad_frac) * float(params.R))

        opts_common = dict(
            dpi=300,
            show=True,
            robust=True,
            robust_pct=97.0,
            n_levels=30,
            n_line_levels=20,
            x_scale=1e3,
            y_scale=1e3,
            x_label="x [mm]",
            y_label="y [mm]",
            value_scale=1e-6,
            cbar_label="Stress [MPa]",
            cmap="jet",
        )

        xs, ys, Sxx, Syy, Sxy, _ = eval_and_plot_stress_components_contours_separate(
            solver,
            bbox=bbox,
            out_dir=out_dir,
            basename=basename,
            n=int(params.n_grid),
            inside=inside,
            normalize_by=None,
            opts_xx=ContourOpts(**opts_common, title=r"$\sigma_{xx}$", symmetric=True),
            opts_yy=ContourOpts(**opts_common, title=r"$\sigma_{yy}$", symmetric=True),
            opts_xy=ContourOpts(**opts_common, title=r"$\sigma_{xy}$", symmetric=True),
        )

        # Save with canonical names expected by run_network_growth_uncoupled
        np.save(out_dir / "xs.npy", xs)
        np.save(out_dir / "ys.npy", ys)
        np.save(out_dir / "Sxx.npy", Sxx)
        np.save(out_dir / "Syy.npy", Syy)
        np.save(out_dir / "Sxy.npy", Sxy)

        return xs, ys, Sxx, Syy, Sxy


    solver_bem, mesh_bem = solve_disk_imposed_uy(
        disk,
        uy_top=float(U_Y_TOP_IMPOSED),
        uy_bottom=float(U_Y_BOTTOM_IMPOSED),
    )
    _ = save_bem_field_arrays_and_plots(solver_bem, disk, BEM_DIR)

    comp_ref = compute_disk_boundary_compliance_from_solver(
        solver=solver_bem,
        mesh=mesh_bem,
        h=float(disk.h),
        arc_half_angle_deg=float(disk.arc_half_angle_deg),
    )
    C_REL_REF = float(comp_ref.get("C_rel_m_per_N", np.nan))
    print("[bem] reference C_rel_m_per_N =", C_REL_REF)

    # Mutable compliance state updated after each outer-cycle BEM update.
    _compliance_state = {"C_rel": float(C_REL_REF)}


    def compliance_callback(**kwargs):
        return float(_compliance_state["C_rel"])


    # -----------------------
    # Crack growth run + metrics save
    # -----------------------
    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=BEM_DIR,
        combined_out_dir=RUN_DIR,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )

    # Mode switch
    #   "no_coupling" : single growth run using fixed BEM field
    #   "iterative"   : outer-loop traction feedback update of BEM
    COUPLING_METHOD = "iterative"
    OUTER_CYCLES = 8
    n_inner_max_for_index = 8


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0, 2), int)
        return V, C


    def _cycle_dir_sort_key(p: Path):
        name = p.name
        idx = -1
        if name.startswith("cycle_"):
            parts = name.split("_")
            if len(parts) > 1 and parts[1].isdigit():
                idx = int(parts[1])

        phase_rank = 0
        if "terminated" in name:
            phase_rank = 4
        elif "post_simplify" in name:
            phase_rank = 3
        elif "pre_simplify" in name:
            phase_rank = 2
        elif "initial" in name:
            phase_rank = 1
        return (idx, phase_rank, name)


    def _latest_cycle_plot_dir(outer_dir: Path) -> Path:
        candidates = []
        for p in outer_dir.glob("cycle_*"):
            if p.is_dir() and (p / "network_graph.png").exists() and (p / "deformed_network.png").exists():
                candidates.append(p)
        if not candidates:
            return outer_dir
        return sorted(candidates, key=_cycle_dir_sort_key)[-1]


    def _save_outer_boundary_snapshot_to_cycle_dir(*, source_bem_dir: Path, outer_dir: Path, outer_cycle: int):
        dst = _latest_cycle_plot_dir(outer_dir)
        file_map = [
            ("xs.npy", "outer_boundary_xs.npy"),
            ("ys.npy", "outer_boundary_ys.npy"),
            ("Sxx.npy", "outer_boundary_Sxx.npy"),
            ("Syy.npy", "outer_boundary_Syy.npy"),
            ("Sxy.npy", "outer_boundary_Sxy.npy"),
            ("brazilian_disk_iter_sxx_contour.png", "outer_boundary_sxx_contour.png"),
            ("brazilian_disk_iter_syy_contour.png", "outer_boundary_syy_contour.png"),
            ("brazilian_disk_iter_sxy_contour.png", "outer_boundary_sxy_contour.png"),
        ]

        copied = []
        for src_name, dst_name in file_map:
            src = Path(source_bem_dir) / src_name
            if src.exists():
                shutil.copy2(src, dst / dst_name)
                copied.append(dst_name)

        manifest = {
            "outer_cycle": int(outer_cycle),
            "saved_at": datetime.now().isoformat(),
            "source_bem_dir": str(source_bem_dir),
            "destination_dir": str(dst),
            "copied_files": copied,
        }
        (dst / f"outer_boundary_snapshot_outer_{int(outer_cycle):02d}.json").write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"[outer-cycle] saved outer-boundary snapshot to {dst}")


    base_growth_kwargs = dict(
        max_cycles=4,
        L_limit_mm=20.0,
        vertex_high=25,
        max_inner_cycles_per_outer=6,
        enable_inner_cycle_metrics_save=True,
        domain_area_m2=float(np.pi * disk.R * disk.R),
        disk_radius_m=float(disk.R),
        arc_half_angle_deg=float(disk.arc_half_angle_deg),
        boundary_band_frac=0.05,
        loading_axis=(0.0, 1.0),
        control_mode="displacement",
        disp_delta_m=float(abs(U_Y_TOP_IMPOSED - U_Y_BOTTOM_IMPOSED)),
        disk_thickness_m=float(disk.h),
        inner_cycle_max_for_index=int(n_inner_max_for_index),
        compliance_evaluator=compliance_callback,
        check_tip_outside_disk=True,
        tip_outside_tolerance_m=0.0,
        # Match VCM_3_0 kinked-propagation strategy: iterative intersections + conservative simplification.
        intersection_detect_mode="iterative",
        intersection_verbose=False,
        simplify_each_step=False,
        simplification_config={
            "max_angle_deviation": 5.0,
            "min_edge_length": 0.50 * mm,
            "merge_at_degree2": True,
            "remove_degree2_nodes": False,
            "merge_vertex_tolerance": 1.0 * mm,
            "max_merged_edge_length": 20.0 * mm,
            "preserve_tips": True,
            "preserve_junctions": True,
        },
        solver_kwargs={
            "n_crack_elements": 40,
            "Kc_demo": Kc_demo,
            "refine_kinks": False,
            "refine_junction_endpoints": False,
            "endpoint_min_nodes": 4,
            "kink_side_nodes": 1,
            "tip_cluster": "power",
            "tip_cluster_power": 2.5,
            "other_min_panels": 4,
        },
    )

    V_curr = vertices.copy()
    C_curr = connectivity.copy()
    all_rows = []
    res_final = None

    if COUPLING_METHOD == "no_coupling":
        grow = CrackGrowthParams(**base_growth_kwargs)
        res_final = run_network_growth_uncoupled(
            bem_dir=BEM_DIR,
            out_dir=RUN_DIR,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        m = np.load(RUN_DIR / "inner_cycle_metrics.npz", allow_pickle=True)
        nloc = int(np.asarray(m["total_cycle_number"]).size)
        for i in range(nloc):
            row = {k: np.asarray(m[k])[i] for k in m.files}
            all_rows.append(row)

    elif COUPLING_METHOD == "iterative":
        boundary_mesh = build_boundary({"type": "circle", "R": float(disk.R), "n_boundary": int(disk.n_boundary_elements), "center": (0.0, 0.0)})

        for oc in range(1, int(OUTER_CYCLES) + 1):
            print(f"\n[outer-cycle] {oc}/{OUTER_CYCLES} mode=iterative")
            outer_dir = RUN_DIR / f"outer_{oc:02d}"
            outer_dir.mkdir(parents=True, exist_ok=True)

            grow = CrackGrowthParams(**base_growth_kwargs)
            res_outer = run_network_growth_uncoupled(
                bem_dir=BEM_DIR,
                out_dir=outer_dir,
                vertices=V_curr,
                connectivity=C_curr,
                params=grow,
                plot_hook=hook,
            )
            res_final = res_outer

            # Concatenate metrics with continuous total_cycle_number over outer loops
            m = np.load(outer_dir / "inner_cycle_metrics.npz", allow_pickle=True)
            nloc = int(np.asarray(m["total_cycle_number"]).size)
            cycle_offset = (oc - 1) * int(n_inner_max_for_index)
            for i in range(nloc):
                row = {k: np.asarray(m[k])[i] for k in m.files}
                row["total_cycle_number"] = int(row["total_cycle_number"]) + int(cycle_offset)
                row["outer_loop_index"] = int(oc)
                all_rows.append(row)

            # Iterative coupling update: crack-induced boundary tractions -> BEM
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_outer, boundary_mesh=boundary_mesh)
            solver_iter, mesh_iter, _ = solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=BEM_DIR,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                show=False,
            )
            _save_outer_boundary_snapshot_to_cycle_dir(
                source_bem_dir=BEM_DIR,
                outer_dir=outer_dir,
                outer_cycle=oc,
            )

            comp_iter = compute_disk_boundary_compliance_from_solver(
                solver=solver_iter,
                mesh=mesh_iter,
                h=float(disk.h),
                arc_half_angle_deg=float(disk.arc_half_angle_deg),
            )
            _compliance_state["C_rel"] = float(comp_iter.get("C_rel_m_per_N", np.nan))
            print(f"[outer-cycle] updated C_rel_m_per_N = {_compliance_state['C_rel']:.6e}")

            # Next outer cycle starts from updated network
            V_curr, C_curr = _network_to_arrays(res_outer.calc.network)

    else:
        raise ValueError("COUPLING_METHOD must be 'no_coupling' or 'iterative'.")


    # Save combined metrics to canonical location for plotting
    if not all_rows:
        raise RuntimeError("No inner-cycle metrics collected.")

    all_keys = sorted({k for r in all_rows for k in r.keys()})
    combined = {}
    for k in all_keys:
        vals = [r.get(k, np.nan) for r in all_rows]
        arr = np.asarray(vals)
        if arr.dtype.kind in ("i", "u", "f", "b"):
            combined[k] = arr
        else:
            combined[k] = arr.astype(str)

    np.savez(RUN_DIR / "inner_cycle_metrics.npz", **combined)


    # -----------------------
    # Load + plot saved inner-cycle metrics vs total cycle number
    # -----------------------
    metrics_npz = RUN_DIR / "inner_cycle_metrics.npz"
    if not metrics_npz.exists():
        raise FileNotFoundError(f"Missing metrics file: {metrics_npz}")

    m = np.load(metrics_npz, allow_pickle=True)
    x = np.asarray(m["total_cycle_number"], int)

    # Print all per-inner-cycle metrics (including K_eff) in table form
    exclude_nonmetrics = {"control_mode"}
    print("\n[inner-cycle metrics]\n")
    metric_cols = [k for k in m.files if k not in exclude_nonmetrics and np.asarray(m[k]).ndim == 1 and np.asarray(m[k]).size == x.size]
    metric_cols = sorted(metric_cols)

    # Keep identifiers first for readability
    front = [k for k in ["total_cycle_number", "outer_loop_index", "outer_cycle", "inner_step", "global_step"] if k in metric_cols]
    rest = [k for k in metric_cols if k not in front]
    cols = front + rest

    print(" | ".join(cols))
    print("-|-".join(["---"] * len(cols)))
    for i in range(x.size):
        vals = []
        for k in cols:
            v = np.asarray(m[k])[i]
            arr_k = np.asarray(m[k])
            if np.issubdtype(arr_k.dtype, np.integer):
                vals.append(f"{int(v)}")
            elif np.issubdtype(arr_k.dtype, np.floating):
                vals.append("nan" if not np.isfinite(v) else f"{float(v):.6e}")
            else:
                vals.append(str(v))
        print(" | ".join(vals))

    # Plot ALL numeric metrics versus total cycle number
    exclude = {
        "total_cycle_number",
        "cycle_index",
        "outer_cycle",
        "inner_step",
        "global_step",
        "control_mode",
    }
    metric_keys = []
    for k in m.files:
        if k in exclude:
            continue
        arr = np.asarray(m[k])
        if arr.ndim == 1 and arr.size == x.size and np.issubdtype(arr.dtype, np.number):
            metric_keys.append(k)

    metric_keys = sorted(metric_keys)
    if not metric_keys:
        raise RuntimeError("No numeric metric arrays found to plot.")

    n = len(metric_keys)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.6*nrows), constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()

    for i, k in enumerate(metric_keys):
        ax = axs[i]
        y = np.asarray(m[k], float)
        ax.plot(x, y, "-o", ms=3)
        ax.set_xlabel("total_cycle_number")
        ax.set_ylabel(k)
        ax.set_title(k)
        ax.grid(True, alpha=0.3)

    for j in range(len(metric_keys), len(axs)):
        axs[j].set_axis_off()

    plt.show()
    print("[done] disk energetics workflow complete")




    # ---- next cell ----

    # Extract alpha, Q, Keffe_mean vs total cycle number and plot/save them
    from pathlib import Path
    import re
    import numpy as np
    import matplotlib.pyplot as plt

    # Resolve run directory and metrics path with fallback when this cell is
    # executed in isolation. `globals()` is the wrong probe when these cells
    # are bundled into a wrapping function (RUN_DIR is then a local of the
    # wrapper, not a module global) — use try/except instead so the bundled
    # form picks up the RUN_DIR from the preceding cell.
    try:
        run_dir = Path(RUN_DIR)
    except NameError:
        cwd = Path.cwd().resolve()
        repo_root_guess = cwd.parent if cwd.name == "notebooks" else cwd
        base = repo_root_guess / "output" / "NetworkSimulations"
        candidates = sorted([p for p in base.glob("*") if p.is_dir()])
        if not candidates:
            raise FileNotFoundError(f"No run directories found in {base}")
        run_dir = candidates[-1]

    metrics_npz = run_dir / "inner_cycle_metrics.npz"


    def _rows_to_metrics(rows):
        keys = sorted({k for r in rows for k in r.keys()})
        out = {}
        for k in keys:
            vals = [r.get(k, np.nan) for r in rows]
            arr = np.asarray(vals)
            if arr.dtype.kind in ("i", "u", "f", "b"):
                out[k] = arr
            else:
                out[k] = arr.astype(str)
        return out


    def _build_from_outer_dirs(base_dir):
        rows = []
        running_total = 0
        outer_dirs = sorted([p for p in base_dir.glob("outer_*") if p.is_dir()])

        for od in outer_dirs:
            p = od / "inner_cycle_metrics.npz"
            if not p.exists():
                continue

            mloc = np.load(p, allow_pickle=True)
            if "total_cycle_number" not in mloc.files:
                continue

            nloc = int(np.asarray(mloc["total_cycle_number"]).size)
            if nloc == 0:
                continue

            local_total = np.asarray(mloc["total_cycle_number"], int)
            local_total = local_total - int(np.min(local_total)) + 1

            mo = re.search(r"outer_(\d+)", od.name)
            oc = int(mo.group(1)) if mo else np.nan

            for i in range(nloc):
                row = {k: np.asarray(mloc[k])[i] for k in mloc.files}
                row["total_cycle_number"] = int(local_total[i]) + int(running_total)
                if "outer_loop_index" not in row:
                    row["outer_loop_index"] = int(oc) if np.isfinite(oc) else np.nan
                rows.append(row)

            running_total = int(max(r["total_cycle_number"] for r in rows))

        return _rows_to_metrics(rows) if rows else None


    m = None
    source_label = None

    # Prefer reconstruction from per-outer files because it is robust to interrupted runs.
    m_outer = _build_from_outer_dirs(run_dir)
    if m_outer is not None and "total_cycle_number" in m_outer:
        m = m_outer
        source_label = "outer_* metrics"
    elif metrics_npz.exists():
        mz = np.load(metrics_npz, allow_pickle=True)
        m = {k: np.asarray(mz[k]) for k in mz.files}
        source_label = str(metrics_npz)
    else:
        raise FileNotFoundError(
            f"No metrics found in {run_dir}. Run the growth workflow cell first."
        )

    x_all = np.asarray(m["total_cycle_number"], int)
    mask = (x_all >= 1) & (x_all <= 75)
    if not np.any(mask):
        raise RuntimeError("No cycles in [1, 75] found for plotting.")

    x = x_all[mask]


    def _pick_metric(candidates, label):
        for k in candidates:
            if k in m:
                return np.asarray(m[k], float), k
        raise KeyError(f"Could not find {label}. Tried {candidates}. Available keys: {list(m.keys())}")


    def _pick_keffe_mean():
        explicit = [
            "Keffe_mean",
            "K_effe_mean",
            "K_eff_mean",
            "Keff_mean",
            "K_eff_mean_MPa_sqrt_m",
        ]
        for k in explicit:
            if k in m:
                return np.asarray(m[k], float), k

        for k in m.keys():
            kl = str(k).lower()
            if ("k_eff" in kl or "keff" in kl) and "mean" in kl:
                arr = np.asarray(m[k])
                if arr.ndim == 1 and arr.size == x_all.size and np.issubdtype(arr.dtype, np.number):
                    return np.asarray(arr, float), k

        raise KeyError(
            "Could not find Keffe_mean-like metric. "
            f"Available keys: {list(m.keys())}"
        )


    def _add_stable_crack_annotation(ax, y):
        x_line = 26
        ax.axvline(x_line, color="red", linestyle="--", linewidth=1.5)

        y = np.asarray(y, float)
        y_finite = y[np.isfinite(y)]
        if y_finite.size == 0:
            y_mid = 0.0
        else:
            y_mid = 0.5 * (float(np.min(y_finite)) + float(np.max(y_finite)))

        x0 = x_line + 1.0
        x1 = x_line + 3.0
        ax.annotate(
            "",
            xy=(x1, y_mid),
            xytext=(x0, y_mid),
            arrowprops=dict(arrowstyle="->", color="black", linewidth=3.0),
        )
        ax.text(x1 + 0.3, y_mid, "Stable Crack", color="black", va="center", ha="left")


    alpha_all, alpha_key = _pick_metric(["alpha", "alpha_mean"], "alpha")
    Q_all, Q_key = _pick_metric(["Q", "Q_mean", "Q_total"], "Q")
    Keffe_all, Keffe_key = _pick_keffe_mean()

    alpha = alpha_all[mask]
    Q = Q_all[mask]
    Keffe = Keffe_all[mask]

    # Save directory for plots
    plots_dir = run_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    # 1) Damage Metric alpha
    fig1, ax1 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax1.plot(x, alpha, "-o", ms=3)
    ax1.set_xlabel("Growth Cycle Number")
    ax1.set_ylabel(r"Damage Metric, $\alpha$, m$^{-1}$")
    ax1.set_xlim(1, 75)
    ax1.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax1, alpha)
    fig1.savefig(plots_dir / "damage_metric_alpha_vs_growth_cycle.png", dpi=300)

    # 2) Orientation Factor Q
    fig2, ax2 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax2.plot(x, Q, "-o", ms=3)
    ax2.set_xlabel("Growth Cycle Number")
    ax2.set_ylabel("Orientation Factor, Q")
    ax2.set_xlim(1, 75)
    ax2.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax2, Q)
    fig2.savefig(plots_dir / "orientation_factor_Q_vs_growth_cycle.png", dpi=300)

    # 3) Crack Tip SIF Keff
    fig3, ax3 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax3.plot(x, Keffe, "-o", ms=3)
    ax3.set_xlabel("Growth Cycle Number")
    ax3.set_ylabel(r"Crack Tip SIF, K$_{eff}$, MPa$\sqrt{m}$")
    ax3.set_xlim(1, 75)
    ax3.axhline(1.00, color="black", linestyle="-.", linewidth=1.5)
    ax3.text(27.0, 1.03, "Fracture Toughness", color="black", va="bottom", ha="left")
    ax3.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax3, Keffe)
    fig3.savefig(plots_dir / "crack_tip_sif_keff_vs_growth_cycle.png", dpi=300)

    plt.show()
    print(f"[metrics source] {source_label}")
    print(f"[cycles plotted] min={int(np.min(x))}, max={int(np.max(x))}, count={int(x.size)}")
    print(f"[selected keys] alpha={alpha_key}, Q={Q_key}, Keffe_mean={Keffe_key}")
    print(f"[saved plots] {plots_dir}")

    return locals()


### Case `disk_experiments` (from `disk_experiments.ipynb`)


In [ ]:
def run_disk_experiments():
    """# Disk Experiments

Configurable experiment runner for Brazilian-disk + crack-network coupling.

This notebook:
- exposes run inputs in one place,
- writes all outputs to `output/disk_experiments/<run_id>`,
- writes a provenance file containing run configuration."""
    global BEMBlockV2, BrazilianDiskParams, CrackGrowthParams, MISSING, Path, PlotParams, asdict, build_boundary, compose_solver_kwargs_v2, compute_crack_boundary_tractions_direct, datetime, ensure_bem_field, fields, json, make_standard_plot_hook, np, os, plot_total_field, plt, run_network_growth_uncoupled, solve_bem_with_extra_boundary_tractions, subprocess, sys

    from pathlib import Path
    from datetime import datetime
    import os
    import sys
    import json
    import subprocess
    from dataclasses import asdict, fields, MISSING

    import numpy as np

    # -----------------------
    # Repo setup
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
    from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
    from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.boundary_conditions import build_boundary


    # =======================
    # USER CONFIG (edit here)
    # =======================

    # Run control===
    COUPLING_METHOD = "iterative"   # "no_coupling" | "iterative" | "direct"
    OUTER_CYCLES = 4
    SKIP_BEM_SOLVE = True

    # Initial network geometry
    mm = 1e-3
    # VERTICES = np.array([
    #     [0, 0.0 * mm, -3.0 * mm],
    #     [1, 0.0 * mm,  3.0 * mm],
    # ], float)
    # CONNECTIVITY = np.array([[0, 1]], int)
    VERTICES = np.array([
        [0,   0.0*mm,  0.0*mm],   
        [1,  1.0*mm,  0.0*mm],   
        [2,  2.0*mm,  1*mm],   
        [3,  4.0*mm,  2*mm],   
        [4,  10.0*mm, 2.0*mm],
        [5,  12.0*mm, -1.0*mm],
        [6,  9.0*mm, -2*mm],
        [7,  7.0*mm, -4.0*mm],
        [8,  7.0*mm, -6.0*mm],
        [9, 8.0*mm, -8.0*mm]
    ], dtype=float)

    CONNECTIVITY = np.array([
        [0, 1],   # e0
        [1, 2],   # e1
        [2, 3],
        [3, 4],
        [4, 5],
        [5, 6],
        [6, 7],
        [7, 8],
        [7, 9],
    ], dtype=int)

    # Disk/BEM params (start from module defaults, set explicit current values)
    DISK_PARAMS = dict(
        R=25.4e-3 / 2,
        P_total=3.8e3,
        E=231.52e9,
        nu=0.3,
        n_boundary_elements=120,
        gauss_n=6,
        plane_strain=True,
        arc_half_angle_deg=10.0,
        h=6.35e-3,
        n_grid=120,
        pad_frac=0.03,
    )

    # BEM baseline compute/load behavior
    BEM_FIELD_CONFIG = dict(
        show=True,
        save_contours=True,
    )

    # Plot config
    PLOT_PARAMS = dict(
        cmap="jet",
        n_bands=40,
        vmax_factor=2.0,
        dpi=300,
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        n_levels=60,
        n_line_levels=15,
        fixed_vlim_mpa=None,
        augmented_correction_stride=4,
    )

    HOOK_CONFIG = dict(
        show_initial=True,
        show_cycles=True,
        show_final=True,
        only_post_simplify=True,
    )

    # Solver kwargs exposed (defaults from run_network_growth_uncoupled)
    SOLVER_KWARGS_DEFAULTS = dict(
        n_crack_elements=60,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=9,
        nq_stress=12,
        crack_mode="half",
        junction_model="core",
    )

    # One-way run solver kwargs (editable)
    ONE_WAY_SOLVER_KWARGS = dict(SOLVER_KWARGS_DEFAULTS)

    # Growth / propagation params (including newly used simplification_config)
    GROWTH_PARAMS = dict(
        material_E=200e9,
        material_nu=0.30,
        plane_stress=False,
        f0=0.03,
        f_fixed=0.03,
        step_mode="fixed_step",
        simultaneous_tip_growth=True,
        Kc_demo=1e5,
        rmax_frac=0.25,
        min_pts=8,
        max_cycles=8,
        vertex_high=16,
        L_limit_mm=20.0,
        simplification_config=None,   # e.g. {"max_angle_deviation": 2.0, ...}
    )

    # Direct KKT payload + solver knobs (used only when COUPLING_METHOD == "direct")
    AUGMENTED_CONFIG = dict(
        d_mode="correction_zero",
        payload_gauss_n=4,
        augmented_enforce_crack_equilibrium=True,
        augmented_equilibrate=True,
        augmented_equilibrate_cols=False,
        augmented_physical_scaling=True,
        augmented_scale_traction_rows=True,
        augmented_ridge_q=1e-4,
        augmented_ridge_y=1e-8,
        augmented_keep_bem_applied=True,
    )

    # Iterative BEM update controls
    ITERATIVE_UPDATE_CONFIG = dict(
        show=False,
        gauss_n=None,   # None => disk.gauss_n
    )


    # =======================
    # Utilities
    # =======================

    def _git_short_hash(default="nogit"):
        try:
            out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root)
            return out.decode("utf-8", errors="ignore").strip() or default
        except Exception:
            return default


    def _run_id():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{ts}_{_git_short_hash()}"


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0, 2), int)
        return V, C


    def _save_total_as_bem_arrays(bem_dir: Path, Sxx_tot, Syy_tot, Sxy_tot):
        xs = np.load(bem_dir / "xs.npy")
        ys = np.load(bem_dir / "ys.npy")
        np.save(bem_dir / "xs.npy", xs)
        np.save(bem_dir / "ys.npy", ys)
        np.save(bem_dir / "Sxx.npy", np.asarray(Sxx_tot, float))
        np.save(bem_dir / "Syy.npy", np.asarray(Syy_tot, float))
        np.save(bem_dir / "Sxy.npy", np.asarray(Sxy_tot, float))


    def _dataclass_defaults(dc_type):
        d = {}
        for f in fields(dc_type):
            if f.default is not MISSING:
                d[f.name] = f.default
            elif f.default_factory is not MISSING:  # type: ignore[attr-defined]
                d[f.name] = f.default_factory()      # type: ignore[misc]
            else:
                d[f.name] = None
        return d


    def _fmt(v):
        if isinstance(v, np.ndarray):
            return np.array2string(v, separator=", ")
        if isinstance(v, (list, tuple, dict)):
            try:
                return json.dumps(v, ensure_ascii=False)
            except Exception:
                return str(v)
        return str(v)


    def _write_provenance(run_dir: Path, run_id: str, table_rows):
        prov = run_dir / f"provenance_{run_id}.md"
        lines = []
        lines.append(f"# Provenance - {run_id}\n")
        lines.append("")
        lines.append(f"- Generated: {datetime.now().isoformat()}\n")
        lines.append(f"- Repo root: `{repo_root}`\n")
        lines.append(f"- Notebook: `VCM_3_2/notebooks/disk_experiments.ipynb`\n")
        lines.append("")
        lines.append("## Run Configuration\n")
        lines.append("")
        lines.append("| Group | Parameter | Default | Effective | Source |\n")
        lines.append("|---|---|---|---|---|\n")
        for g, p, d, e, s in table_rows:
            lines.append(f"| {g} | `{p}` | `{_fmt(d)}` | `{_fmt(e)}` | {s} |\n")
        prov.write_text("".join(lines), encoding="utf-8")


    def _write_provenance_json(run_dir: Path, run_id: str, table_rows):
        prov_json = run_dir / f"provenance_{run_id}.json"
        rows = []
        for g, p, d, e, s in table_rows:
            rows.append({
                "group": g,
                "parameter": p,
                "default": d,
                "effective": e,
                "source": s,
            })
        payload = {
            "run_id": run_id,
            "generated_at": datetime.now().isoformat(),
            "repo_root": str(repo_root),
            "notebook": "VCM_3_2/notebooks/disk_experiments.ipynb",
            "rows": rows,
        }

        def _json_default(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, (np.floating, np.integer)):
                return obj.item()
            return str(obj)

        prov_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")


    # =======================
    # Build run directories
    # =======================
    run_id = _run_id()
    run_dir = repo_root / "output" / "disk_experiments" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"[run_id] {run_id}")
    print(f"[run_dir] {run_dir}")


    # =======================
    # Instantiate config objects
    # =======================
    disk = BrazilianDiskParams(**DISK_PARAMS)
    plot_params = PlotParams(**PLOT_PARAMS)

    boundary_mesh = build_boundary({
        "type": "circle",
        "R": disk.R,
        "n_boundary": disk.n_boundary_elements,
        "center": (0.0, 0.0),
    })

    bem_block = BEMBlockV2(disk=disk, out_dir=run_dir)
    one_way_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=ONE_WAY_SOLVER_KWARGS)

    aug_payload = bem_block.build_augmented_payload(
        d_mode=AUGMENTED_CONFIG["d_mode"],
        gauss_n=AUGMENTED_CONFIG["payload_gauss_n"],
    )
    direct_kwargs = compose_solver_kwargs_v2(
        "full_kkt",
        base_solver_kwargs={
            "augmented_enforce_crack_equilibrium": AUGMENTED_CONFIG["augmented_enforce_crack_equilibrium"],
            "augmented_equilibrate": AUGMENTED_CONFIG["augmented_equilibrate"],
            "augmented_equilibrate_cols": AUGMENTED_CONFIG["augmented_equilibrate_cols"],
            "augmented_physical_scaling": AUGMENTED_CONFIG["augmented_physical_scaling"],
            "augmented_scale_traction_rows": AUGMENTED_CONFIG["augmented_scale_traction_rows"],
        },
        augmented_payload=aug_payload,
        augmented_ridge_q=AUGMENTED_CONFIG["augmented_ridge_q"],
        augmented_ridge_y=AUGMENTED_CONFIG["augmented_ridge_y"],
        augmented_keep_bem_applied=AUGMENTED_CONFIG["augmented_keep_bem_applied"],
    )

    hook = make_standard_plot_hook(
        bem_dir=run_dir,
        combined_out_dir=run_dir,
        plot_params=plot_params,
        show_initial=HOOK_CONFIG["show_initial"],
        show_cycles=HOOK_CONFIG["show_cycles"],
        show_final=HOOK_CONFIG["show_final"],
        only_post_simplify=HOOK_CONFIG["only_post_simplify"],
    )


    # =======================
    # Provenance table rows
    # =======================
    rows = []

    # Run control
    rows += [
        ("run", "COUPLING_METHOD", "iterative", COUPLING_METHOD, "notebook"),
        ("run", "OUTER_CYCLES", 5, OUTER_CYCLES, "notebook"),
        ("run", "SKIP_BEM_SOLVE", True, SKIP_BEM_SOLVE, "notebook"),
    ]

    # Network
    rows += [
        ("network", "mm", 1e-3, mm, "notebook"),
        ("network", "vertices", "(current in source)", VERTICES, "notebook"),
        ("network", "connectivity", "(current in source)", CONNECTIVITY, "notebook"),
    ]

    # Disk params defaults vs effective
    disk_defaults = _dataclass_defaults(BrazilianDiskParams)
    for k, v in asdict(disk).items():
        rows.append(("disk", k, disk_defaults.get(k), v, "notebook/default"))

    # BEM field controls
    rows += [
        ("bem_field", "recompute", False, (not SKIP_BEM_SOLVE), "derived"),
        ("bem_field", "show", True, BEM_FIELD_CONFIG["show"], "notebook"),
        ("bem_field", "save_contours", True, BEM_FIELD_CONFIG["save_contours"], "notebook"),
    ]

    # Plot params defaults vs effective
    plot_defaults = PlotParams().__dict__.copy()
    for k, v in plot_params.__dict__.items():
        rows.append(("plot", k, plot_defaults.get(k), v, "notebook/default"))
    for k, v in HOOK_CONFIG.items():
        hook_default = dict(show_initial=True, show_cycles=False, show_final=True, only_post_simplify=True).get(k)
        rows.append(("plot_hook", k, hook_default, v, "notebook/default"))

    # Solver kwargs
    for k, v in SOLVER_KWARGS_DEFAULTS.items():
        rows.append(("solver_kwargs", k, v, ONE_WAY_SOLVER_KWARGS.get(k, v), "notebook/default"))

    # Growth params
    growth_defaults = _dataclass_defaults(CrackGrowthParams)
    for k, v in GROWTH_PARAMS.items():
        rows.append(("growth", k, growth_defaults.get(k), v, "notebook/default"))

    # Direct / augmented
    rows += [
        ("augmented", "d_mode", "correction_zero", AUGMENTED_CONFIG["d_mode"], "notebook/default"),
        ("augmented", "payload_gauss_n", 4, AUGMENTED_CONFIG["payload_gauss_n"], "notebook/default"),
        ("augmented", "augmented_ridge_q", 1e-4, AUGMENTED_CONFIG["augmented_ridge_q"], "notebook/default"),
        ("augmented", "augmented_ridge_y", 1e-10, AUGMENTED_CONFIG["augmented_ridge_y"], "notebook/default"),
        ("augmented", "augmented_keep_bem_applied", True, AUGMENTED_CONFIG["augmented_keep_bem_applied"], "notebook/default"),
    ]

    # Iterative update
    rows += [
        ("iterative_update", "show", False, ITERATIVE_UPDATE_CONFIG["show"], "notebook/default"),
        ("iterative_update", "gauss_n", "disk.gauss_n", ITERATIVE_UPDATE_CONFIG["gauss_n"], "notebook/default"),
    ]

    _write_provenance(run_dir, run_id, rows)
    _write_provenance_json(run_dir, run_id, rows)
    print(f"[provenance] {run_dir / ('provenance_' + run_id + '.md')}")
    print(f"[provenance-json] {run_dir / ('provenance_' + run_id + '.json')}")


    # =======================
    # Baseline BEM field
    # =======================
    ensure_bem_field(
        disk,
        run_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=bool(BEM_FIELD_CONFIG["show"]),
        save_contours=bool(BEM_FIELD_CONFIG["save_contours"]),
    )


    # =======================
    # Outer-cycle run
    # =======================
    V_curr = VERTICES.copy()
    C_curr = CONNECTIVITY.copy()
    res_last = None

    for cyc in range(1, int(OUTER_CYCLES) + 1):
        print(f"\n[outer-cycle] {cyc}/{OUTER_CYCLES} method={COUPLING_METHOD}")

        grow = CrackGrowthParams(
            material_E=GROWTH_PARAMS["material_E"],
            material_nu=GROWTH_PARAMS["material_nu"],
            plane_stress=GROWTH_PARAMS["plane_stress"],
            solver_kwargs=one_way_kwargs,
            f0=GROWTH_PARAMS["f0"],
            f_fixed=GROWTH_PARAMS["f_fixed"],
            step_mode=GROWTH_PARAMS["step_mode"],
            simultaneous_tip_growth=GROWTH_PARAMS["simultaneous_tip_growth"],
            Kc_demo=GROWTH_PARAMS["Kc_demo"],
            rmax_frac=GROWTH_PARAMS["rmax_frac"],
            min_pts=GROWTH_PARAMS["min_pts"],
            max_cycles=GROWTH_PARAMS["max_cycles"],
            vertex_high=GROWTH_PARAMS["vertex_high"],
            L_limit_mm=GROWTH_PARAMS["L_limit_mm"],
            simplification_config=GROWTH_PARAMS["simplification_config"],
        )

        out_growth = run_dir / f"outer_{cyc:02d}_growth"
        out_growth.mkdir(parents=True, exist_ok=True)

        res_growth = run_network_growth_uncoupled(
            bem_dir=run_dir,
            out_dir=out_growth,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        if COUPLING_METHOD == "iterative":
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_growth, boundary_mesh=boundary_mesh)
            solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=run_dir,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                gauss_n=ITERATIVE_UPDATE_CONFIG["gauss_n"],
                show=bool(ITERATIVE_UPDATE_CONFIG["show"]),
            )
            res_last = res_growth

        elif COUPLING_METHOD == "direct":
            grow_direct = CrackGrowthParams(
                material_E=GROWTH_PARAMS["material_E"],
                material_nu=GROWTH_PARAMS["material_nu"],
                plane_stress=GROWTH_PARAMS["plane_stress"],
                solver_kwargs=direct_kwargs,
                f0=GROWTH_PARAMS["f0"],
                f_fixed=GROWTH_PARAMS["f_fixed"],
                step_mode=GROWTH_PARAMS["step_mode"],
                simultaneous_tip_growth=GROWTH_PARAMS["simultaneous_tip_growth"],
                Kc_demo=GROWTH_PARAMS["Kc_demo"],
                rmax_frac=GROWTH_PARAMS["rmax_frac"],
                min_pts=GROWTH_PARAMS["min_pts"],
                max_cycles=0,
                vertex_high=GROWTH_PARAMS["vertex_high"],
                L_limit_mm=GROWTH_PARAMS["L_limit_mm"],
                simplification_config=GROWTH_PARAMS["simplification_config"],
            )

            out_direct = run_dir / f"outer_{cyc:02d}_direct"
            out_direct.mkdir(parents=True, exist_ok=True)

            Vg, Cg = _network_to_arrays(res_growth.calc.network)
            res_direct = run_network_growth_uncoupled(
                bem_dir=run_dir,
                out_dir=out_direct,
                vertices=Vg,
                connectivity=Cg,
                params=grow_direct,
                plot_hook=None,
            )

            Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
                bem_dir=run_dir,
                res=res_direct,
                out_dir=out_direct,
                tag=f"outer_{cyc:02d}_direct",
                params=plot_params,
                show=True,
            )
            _save_total_as_bem_arrays(run_dir, Sxx_tot, Syy_tot, Sxy_tot)
            res_last = res_direct

        else:
            # no_coupling
            res_last = res_growth

        V_curr, C_curr = _network_to_arrays(res_growth.calc.network)

    print("\n[done] outer-cycle run completed")
    print(f"[output] {run_dir}")


    # ---- next cell ----

    ## Least squares fits to K_I and K_II data in compression experiments,
    ## annotated with three propagation regimes and a crack-arrest indicator.

    import os
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Digitized data: y-position (mm) vs SIF (MPa sqrt(m))
    ki = np.array([
        [2.96997115830967, 2.26123418082751],
        [3.16265495749845, 2.16184830336284],
        [3.42439900461923, 2.06238632645141],
        [3.65855700229058, 1.96846438926389],
        [3.94864809332835, 1.97365437153272],
        [4.27987093533829, 1.93472189457183],
        [4.61185477181580, 2.00598141651510],
        [4.88824796243731, 2.02771541850891],
        [5.35812399643854, 2.06576514188742],
        [5.70391988250245, 2.13700944394134],
        [6.25674431346883, 2.18598704787416],
        [6.12349418219729, 2.89136803975435],
        [6.67647081205718, 2.96238404346800],
        [7.13124110587715, 2.81312258859878],
        [7.64285768642461, 2.89520345187091],
        [7.65784927743574, 3.06598583028301],
        [8.19564406766762, 2.93867145585852],
        [8.87224424878430, 2.91037768155426],
        [8.87722876254689, 3.63213527437656],
        [9.49998858508298, 3.80775757760241],
        [10.2042889648192, 3.79045256340986],
        [10.8784539636397, 3.40954439261227],
        [11.0784433097171, 2.36800170462761],
        [11.8003607113776, 0.901641465066549],
        [12.4098031307312, -2.85109621253053],
        [12.3917675618498, -3.46264658655932],
    ])

    kii = np.array([
        [3.32926606831747, 0.0832458032743578],
        [3.58270489895781, 0.0208719880749469],
        [3.92133615945356, 0.0387847263716013],
        [4.13354675992024, 0.130247872547848],
        [4.23163283918159, 0.0388539118074381],
        [4.82452057637757, 0.110335046197003],
        [5.26117502028391, 0.0279351921153762],
        [5.65586220777015, -0.00542162233557428],
        [6.12176322228022, 0.0593422351927445],
        [6.02276515318284, 0.0214160371840272],
        [6.61709949494632, 0.298025699244621],
        [7.58851018598302, 0.0440616882504263],
        [7.06567270257181, -0.0942934594604793],
        [7.57360386935274, -0.0696540099249649],
        [8.12435138653902, 0.0263439270911302],
        [8.15183687332146, -0.0762140471593089],
        [8.81513651544407, -0.0203247941733281],
        [8.81463334863798, -0.0916738472762952],
        [9.47860912115627, 0.0600906958167973],
        [9.47961545476844, 0.202788802022731],
        [10.1417986955400, 0.100372343436504],
        [10.8779789047316, 0.490726006805331],
        [11.6537677759398, 0.497587944123327],
        [12.3629027693043, -0.947069996792312],
        [12.4129993144352, -1.84337989722818],
    ])

    def polyfit_with_bounds(x, y, degree=3):
        coeff = np.polyfit(x, y, degree)
        model = np.poly1d(coeff)
        y_fit = model(x)
        residual = y - y_fit
        lower_shift = residual.min()
        upper_shift = residual.max()
        return model, lower_shift, upper_shift

    ki_x, ki_y = ki[:, 0], ki[:, 1]
    kii_x, kii_y = kii[:, 0], kii[:, 1]

    ki_model, ki_low, ki_up = polyfit_with_bounds(ki_x, ki_y, degree=3)
    kii_model, kii_low, kii_up = polyfit_with_bounds(kii_x, kii_y, degree=3)

    x_min = min(ki_x.min(), kii_x.min())
    x_max = max(ki_x.max(), kii_x.max())
    x_plot = np.linspace(x_min, x_max, 500)

    ki_fit = ki_model(x_plot)
    ki_lower = ki_fit + ki_low
    ki_upper = ki_fit + ki_up

    kii_fit = kii_model(x_plot)
    kii_lower = kii_fit + kii_low
    kii_upper = kii_fit + kii_up

    # Locate where the K_I least-squares fit first crosses zero (crack-arrest indicator)
    sign_change_idx = np.where(np.diff(np.sign(ki_fit)) < 0)[0]
    y_arrest = float(x_plot[sign_change_idx[-1]]) if sign_change_idx.size else 12.0

    # Regime boundaries (mm) along the loading axis
    y_a_b = 8.0    # tensile-dominated  -> geometric stress-concentration band
    y_b_c = 11.0   # geometric          -> compressive contact / arrest zone

    fig, ax = plt.subplots(figsize=(10, 6))

    # --- Regime backgrounds (drawn first so data sits on top)
    ax.axvspan(x_min, y_a_b, color="#cfe9ff", alpha=0.35, zorder=0,
               label="(a) tensile regime")
    ax.axvspan(y_a_b, y_b_c, color="#fff1c2", alpha=0.45, zorder=0,
               label="(b) geometric stress-concentration")
    ax.axvspan(y_b_c, x_max, color="#f9c8c2", alpha=0.45, zorder=0,
               label="(c) compressive / arrest")

    # --- K_I
    ax.scatter(ki_x, ki_y, s=35, color="tab:blue", alpha=0.9, zorder=3, label=r"$K_I$ data")
    ax.plot(x_plot, ki_fit, color="tab:blue", lw=2.2, zorder=4, label=r"$K_I$ least-squares fit")
    ax.plot(x_plot, ki_lower, color="tab:blue", lw=1.0, alpha=0.55, zorder=2)
    ax.plot(x_plot, ki_upper, color="tab:blue", lw=1.0, alpha=0.55, zorder=2)
    ax.fill_between(x_plot, ki_lower, ki_upper, color="lightgreen", alpha=0.18, zorder=1)

    # --- K_II
    ax.scatter(kii_x, kii_y, s=35, color="tab:orange", alpha=0.9, zorder=3, label=r"$K_{II}$ data")
    ax.plot(x_plot, kii_fit, color="tab:orange", lw=2.2, zorder=4, label=r"$K_{II}$ least-squares fit")
    ax.plot(x_plot, kii_lower, color="tab:orange", lw=1.0, alpha=0.55, zorder=2)
    ax.plot(x_plot, kii_upper, color="tab:orange", lw=1.0, alpha=0.55, zorder=2)
    ax.fill_between(x_plot, kii_lower, kii_upper, color="lightgreen", alpha=0.18, zorder=1)

    # --- Zero line and crack-arrest indicator
    ax.axhline(0.0, color="0.35", lw=0.8, zorder=2)
    ax.axvline(y_arrest, color="firebrick", lw=1.6, ls="--", zorder=4)
    y_top = max(ki_y.max(), kii_y.max()) + 0.5
    ax.annotate(
        "crack-arrest indicator\n(" + r"$K_I$ first crosses $0$)",
        xy=(y_arrest, 0.0),
        xytext=(y_arrest - 1.6, y_top - 0.5),
        arrowprops=dict(arrowstyle="->", color="firebrick", lw=1.2),
        color="firebrick", fontsize=14, ha="center",
    )

    # --- Regime labels at top of axes
    ymin, ymax = ax.get_ylim()
    y_label = ymax - 0.05 * (ymax - ymin)
    ax.text(0.5 * (x_min + y_a_b), y_label, "(a)", ha="center", va="top", fontsize=14, color="0.25")
    ax.text(0.5 * (y_a_b + y_b_c), y_label, "(b)", ha="center", va="top", fontsize=14, color="0.25")
    ax.text(0.5 * (y_b_c + x_max), y_label, "(c)", ha="center", va="top", fontsize=14, color="0.25")

    ax.set_xlabel("Crack-tip position along loading axis (mm)", fontsize=16)
    ax.set_ylabel(r"Stress intensity factor (SIF), MPa$\sqrt{\mathrm{m}}$", fontsize=16)
    ax.tick_params(axis="both", which="major", labelsize=14)
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, frameon=True, fontsize=11, loc="lower left")

    plt.tight_layout()

    # Save the annotated figure to disk so it can be included in the manuscript
    out_path = Path(os.getcwd()).resolve() / "SIF_vs_position_regimes.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}  |  K_I sign change at y = {y_arrest:.2f} mm")

    plt.show()

    return locals()


## 4. Dispatch


In [ ]:
CASES = {
    "disk_compression_2": run_disk_compression_2,
    "disk_compression_inclined": run_disk_compression_inclined,
    "disk_energetics": run_disk_energetics,
    "disk_experiments": run_disk_experiments,
}

if CASE not in CASES:
    raise ValueError(f"Unknown CASE={CASE!r}. Expected one of {list(CASES)}.")

if DRY_RUN:
    print(f'DRY RUN — case functions defined but not executed.')
    print(f'Set DRY_RUN = False (in the selector cell) and re-run to invoke run_{CASE}().')
    result = None
else:
    result = CASES[CASE]()
    if isinstance(result, dict):
        print(f'Finished case {CASE!r}; returned-locals keys: {list(result)[:8]}...')
    else:
        print(f'Finished case {CASE!r}.')
